# ANÁLISIS DE LOS COSTOS DE PROCESO DE PRODUCCIÓN

## Extracción y limpieza de los datos

In [ ]:
# Importamos las librerías necesarias para el desarrollo del código
import pandas as pd
import plotly.express as px
import re
import unicodedata

# Cargamos el archivo Excel y leemos la hoja de costos
xls = pd.ExcelFile('jjjjj.xlsx')
df_costos = pd.read_excel('jjjjj.xlsx', sheet_name='Sheet1')
#print(df_costos.columns)

In [ ]:
# Definimos la lista de categorías oficiales para la clasificación de productos
categorias_oficiales = [
    'Accesorios', 'Bata', 'Bermuda', 'Blusa', 'Bolígrafo', 'Botas', 'Botella', 'Botón', 'Broche', 'Buzo', 'Camibuzo', 
    'Camiseta', 'Camisa', 'Cachuchera', 'Casco', 'Chaleco', 'Chaqueta', 'Cinta', 'Cofia', 'Conjunto', 'Cordón', 
    'Cremallera', 'Cuello', 'Delantal', 'Diseño', 'Embone', 'Falda', 'Gafas seguridad', 'Gorra', 'Gorro', 'Guantes', 
    'Guata', 'Hilo', 'Hoodie', 'Impermeable', 'Jean', 'Libreta', 'Lonchera', 'Mangas', 'Marquilla', 'Máscara', 'Medias', 
    'Morral', 'Ojalete', 'Overol', 'Pantalón', 'Pantaloneta', 'Pañoleta', 'Paraguas', 'Pavas', 'Polainas', 'Resorte', 
    'Ropa interior', 'Saco', 'Sastre', 'Servicios', 'Sesgo', 'Sudadera', 'Suéter', 'Tankas', 'Tapabocas', 
    'Tapaoídos', 'Tecnología', 'Termo', 'Uniformes', 'Velcro', 'Zapatos', 'Escafandra'
]

# Creamos una función para normalizar el texto, eliminando acentos y convirtiendo a minúsculas
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

# Creamos una función que clasifique los productos en las categorías oficiales.
def clasificar(Producto):
    producto_norm = normalizar(Producto)
    # Recorremos cada categoría oficial y verificamos si alguna de las palabras está presente en el nombre del producto
    for cat in categorias_oficiales:
        cat_norm = normalizar(cat)
        pattern = r'\b' + re.escape(cat_norm) + r'\b'
        # asignamos la categoría correspondiente si encontramos una coincidencia
        if re.search(pattern, producto_norm):
            return cat
    # Si no encontramos coincidencia, asignamos 'Otros'
    return 'Otros'

df_costos['Categoría'] = df_costos['Producto'].apply(clasificar)   
#print(df_costos['Categoría'].value_counts())

## BOXPLOT de la diferencia de costos en cada proceso de producción

In [ ]:
df_costos.columns = df_costos.columns.str.strip()
figboxplot = px.box(df_costos, 
             x='Proceso Producción', 
             y='Diferencia valor unitario', 
             points='all',
             notched=True,
             color='Proceso Producción',
             template='plotly_dark')
figboxplot.show()

## Visualización de datos del valor costeado por categoría

In [ ]:
# Agrupamos por Categoría y proceso de producción, calculando la desviación estándar y el promedio del valor costeado
df_calc_pag = df_costos.groupby(['Categoría', 'Proceso Producción'])['costo_antes_iva'].agg(['std', 'mean']).reset_index()

# Calculamos el coeficiente de variación, es decir, el porcentaje de desviación respecto al promedio.
df_calc_pag['inestabilidad'] = df_calc_pag['std'] / df_calc_pag['mean']

# Utilizamos el calculo de inestabilidad para identificar las 5 categorías con procesos más inestables.
top5_categorias = (df_calc_pag.groupby('Categoría')['inestabilidad']
                   #Sacamos el promedio de inestabilidd solo por categoría, ya que necesitamos identificar que varíen independientemente del proceso.
                   .mean() 
                   .sort_values(ascending=False)
                   .head(5)
                   .index.tolist())


df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

# Centramos los datos en cero restando el promedio a cada valor costeado.
df_filtrado['variacion_neta'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['costo_antes_iva'].transform(lambda x: x - x.mean())

# Gráfica de cajas para visualizar la variación neta de los valores costeados.
fig_cos = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_neta', 
             color='Proceso Producción',
             points='all',
             title='Variación de los valores costeaados (Centrado en Cero)',
             template='plotly_dark')

# Línea de referencia en cero
fig_cos.add_hline(y=0, line_dash="dash", line_color="pink")

fig_cos.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Procesos más Inestables"
)
#fig_os.update_traces(boxmean='sd')
fig_cos.show()

## Visualización de datos del valor costeado por OS

In [ ]:
# Agrupamos por Categoría y proceso de producción, calculando la desviación estándar y el promedio del valor costeado
df_calc_os = df_costos.groupby(['Categoría', 'Proceso Producción'])['OS-Valor Unitario'].agg(['std', 'mean']).reset_index()

df_calc_os['inestabilidad'] = df_calc_os['std'] / df_calc_os['mean']

# Utilizamos el calculo de inestabilidad para identificar las 5 categorías con procesos más inestables.
top5_categorias = (df_calc_os.groupby('Categoría')['inestabilidad']
                   #Sacamos el promedio de inestabilidd solo por categoría, ya que necesitamos identificar que varíen independientemente del proceso.
                   .mean() 
                   .sort_values(ascending=False)
                   .head(5)
                   .index.tolist())


df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

# Centramos los datos: Restamos el promedio específico de cada Categoría-Proceso
# para que el "0" sea el punto de equilibrio de cada uno.
df_filtrado['variacion_netaos'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['OS-Valor Unitario'].transform(lambda x: x - x.mean())

# --- PASO 4: Gráfica de "Inestabilidad Pura" ---
fig_os = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_netaos', 
             color='Proceso Producción',
             points='all',
             title='Inestabilidad por OS (Centrado en Cero)',
             template='plotly_dark')

# Línea de referencia en cero
fig_os.add_hline(y=0, line_dash="dash", line_color="blue")

fig_os.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Procesos más Inestables"
)

fig_os.show()

## Visualización de la diferencia entre los costeos

In [ ]:
# Agrupamos por Categoría y proceso de producción, calculando la desviación estándar y el promedio del valor costeado
df_calc_dif = df_costos.groupby(['Categoría', 'Proceso Producción'])['Diferencia valor unitario'].agg(['std', 'mean']).reset_index()

df_calc_dif['inestabilidad'] = df_calc_dif['std'] / df_calc_dif['mean']

# Utilizamos el calculo de inestabilidad para identificar las 5 categorías con procesos más inestables.
top5_categorias = (df_calc_dif.groupby('Categoría')['inestabilidad']
                   #Sacamos el promedio de inestabilidd solo por categoría, ya que necesitamos identificar que varíen independientemente del proceso.
                   .mean() 
                   .sort_values(ascending=False)
                   .head(10)
                   .index.tolist())


df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

# Centramos los datos: Restamos el promedio específico de cada Categoría-Proceso
# para que el "0" sea el punto de equilibrio de cada uno.
df_filtrado['variacion_netadif'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['Diferencia valor unitario'].transform(lambda x: x - x.mean())

# --- PASO 4: Gráfica de "Inestabilidad Pura" ---
fig_dif = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_netadif', 
             color='Proceso Producción',
             points='all',
             template='plotly_dark')

# Línea de referencia en cero
fig_dif.add_hline(y=0, line_dash="dash", line_color="blue")

fig_dif.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Procesos más Inestables"
)

fig_dif.show()

## Valor costeado vs pagado en margen de ahorro o sobrecosto

In [ ]:
# Sacamos la desviación en valor absoluto para identificar el impacto económico real. 
df_costos['desviacion_valor'] = df_costos['valor total op'] - df_costos['Valor total OS']

# Columnaq de estado para identificar si es sobrecosto o ahorro.
df_costos['estado'] = df_costos.apply(
    lambda x: 'Ahorro'
    if x['Valor total OS'] < x['valor total op'] 
    else 'Sobrecosto',
    axis=1
)

df_impacto = df_costos.groupby(['Categoría', 'Proceso Producción', 'estado']).agg({
    'desviacion_valor': 'median'
}).reset_index()


df_impacto['abs_impacto'] = df_impacto['desviacion_valor'].abs()
# Top 5 real
top5 = df_impacto.sort_values(by='abs_impacto', ascending=False).head(12)
top5['Etiqueta'] = top5['Categoría'] + ' - ' + top5['Proceso Producción']

fig = px.bar(
    top5,
    x='desviacion_valor',  # 👈 mantiene signo
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={
        'Ahorro': 'green',
        'Sobrecosto': 'red'
    },
    template='plotly_dark',
    #title='Top impacto económico (Ahorro vs Sobrecosto)'
)

fig.add_vline(x=0, line_width=2, line_color="black")

fig.show()

In [ ]:
# Filtramos el DF para obtener solo los datos de confección.
df_conf = df_costos[df_costos['Proceso Producción'] == 'Confección'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_conf['estado'] = df_conf.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)


fig_confe = px.scatter(
    df_conf,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    template='plotly_dark',
    #color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    size=abs(df_conf['costo_antes_iva'] - df_conf['OS-Valor Unitario']),
    title='Costeado vs Pagado - Confección (tamaño = impacto)'
)

min_val = min(df_conf['costo_antes_iva'].min(), df_conf['OS-Valor Unitario'].min())
max_val = max(df_conf['costo_antes_iva'].max(), df_conf['OS-Valor Unitario'].max())

fig_confe.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_confe.show()


In [ ]:
# Filtramos el DF para obtener solo los datos de sublimación.
df_subl = df_costos[df_costos['Proceso Producción'] == 'Sublimación'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_subl['estado'] = df_subl.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)


fig_subl= px.scatter(
    df_subl,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    template='plotly_dark',
    #color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    size=abs(df_subl['costo_antes_iva'] - df_subl['OS-Valor Unitario']),
    title='Costeado vs Pagado - Sublimación (tamaño = impacto)'
)

min_val = min(df_subl['costo_antes_iva'].min(), df_subl['OS-Valor Unitario'].min())
max_val = max(df_subl['costo_antes_iva'].max(), df_subl['OS-Valor Unitario'].max())

fig_subl.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_subl.show()

In [ ]:
# Filtramos el DF para obtener solo los datos de Bordado.
df_bord = df_costos[df_costos['Proceso Producción'] == 'Bordado'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_bord['estado'] = df_bord.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)


fig_bord= px.scatter(
    df_bord,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    template='plotly_dark',
    #color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    size=abs(df_bord['costo_antes_iva'] - df_bord['OS-Valor Unitario']),
    title='Costeado vs Pagado - Bordado (tamaño = impacto)'
)

min_val = min(df_bord['costo_antes_iva'].min(), df_bord['OS-Valor Unitario'].min())
max_val = max(df_bord['costo_antes_iva'].max(), df_bord['OS-Valor Unitario'].max())

fig_bord.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_bord.show()

## INFORME EN HTML

In [ ]:
import base64

with open("logo.png", "rb") as image_file:
    logo_base64 = base64.b64encode(image_file.read()).decode()

In [ ]:
import plotly
print(plotly.__version__)

6.6.0


In [ ]:
import plotly.io as pio

for f_fig in [figboxplot, fig_cos, fig_os, fig_dif, fig, fig_confe, fig_bord, fig_subl]:
    f_fig.update_layout(
        autosize=True,
        height=450,
        width=None,   # 👈 esto es clave, elimina el ancho fijo
        margin=dict(l=40, r=40, t=60, b=40)
    )

In [ ]:
import os
from plotly.io import write_html

def export_reporte():
    output_dir = "./"
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, "reporte.html"), "w", encoding="utf-8") as f:

        # ── HEAD + ESTILOS ──────────────────────────────────────────
        f.write(f"""<!DOCTYPE html>
<html lang='es'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Informe de Costos</title>
    <style>
        body {{
            font-family: "Times New Roman", serif;
            margin: 40px;
            background-color: #000000;
            color: #673ded;
        }}
        .header {{
            position: relative;
            padding: 20px 0;
            border-bottom: 2px solid #ccc;
        }}
        .title {{
            font-size: 32px;
            font-weight: bold;
            color: #B8A6E3;
            text-align: center;
        }}
        .logo {{
            position: absolute;
            top: 50%;
            transform: translateY(-50%);
            height: 60px;
        }}
        .logo.left {{ left: 10px; }}
        .logo.right {{ right: 10px; }}
        .row {{
            display: flex;
            gap: 30px;
            flex-wrap: wrap;
            margin-top: 30px;
        }}
                
        .card .plotly-graph-div {{
            width: 100% !important;
            min-width: 0 !important;
        }}
                
        .card {{
            flex: 1 1 45%;
            min-width: 400px;
            background-color: #181818;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0px 2px 8px rgba(0,0,0,0.1);
        }}
        .card h3 {{
            text-align: center;
            font-size: 25px;
            color: #B8A6E3;
        }}
        .card p {{
            margin-top: 15px;
            font-size: 20px;
            color: #EEBEF8;
            text-align: center;
        }}
    </style>
    
    <script>
        window.addEventListener('load', function() {{
        setTimeout(function() {{
        var graficas = document.querySelectorAll('.plotly-graph-div');
        graficas.forEach(function(g) {{
            Plotly.relayout(g, {{autosize: true}});
            }});
        }}, 300);
    }});
    </script>
</head>
<body>

<div class="header">
    <img src="data:image/png;base64,{logo_base64}" class="logo left">
    <div class="title">INFORME DE COSTOS DE SERVICIOS MOLT</div>
    <img src="data:image/png;base64,{logo_base64}" class="logo right">
</div>
""")

        # ── FILA 1 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>Diferencia de costos por proceso de producción</h3>")
        write_html(figboxplot, file=f, full_html=False,config={"responsive": True}, include_plotlyjs='cdn')  # 👈 solo la primera lleva 'cdn'
        f.write("<p>Este gráfico muestra la variación del valor costeado en cada uno de los procesos de producción.</p></div>")

        f.write("<div class='card'><h3>Valor inicial (costeado) por categoría</h3>")
        write_html(fig_cos, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')  # 👈 las demás en False
        f.write("<p>Aquí podemos visualizar la variación de los valores costeados en las categorías con mayor inestabilidad," \
        " también, cómo se comportan los procesos de producción dentro de cada categoría.</p></div>")

        f.write("</div>")  # cierra row

        # ── FILA 2 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>Valor final (pagado) por categoría</h3>")
        write_html(fig_os, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>En este gráfico se muestra la variación de los valores de OS (pagados) en las categorías con mayor inestabilidad,"
        " también, cómo se comportan los procesos de producción dentro de cada categoría.</p></div>")

        f.write("<div class='card'><h3>Diferencia de costos por categoría</h3>")
        write_html(fig_dif, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>Para este gráfico tomamos la diferencia de valor unitario, es decir, la diferencia entre el valor costeado y pagado, " \
        "así, calculamos las categorías con mayor inestabilidad en este valor," \
        "esta gráfica nos permite comparar con las demás qué tanto varía el valor unitario en los procesos de producción de las categorías mencionadas. </p></div>")

        f.write("</div>")

        # ── FILA 3 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>Impacto económico teniendo en cuenta la diferencia en valores totales</h3>")
        write_html(fig, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>Visualizamos el impacto económico que presenta para la compañía la diferencia de digitación entre los valores costeados y pagados.</p></div>")

        f.write("<div class='card'><h3>Relación entre costos inicial y final en todas las categorías del proceso de confección</h3>")
        write_html(fig_confe, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>Relación entre el valor costeado y pagado en confección. El tamaño del punto indica el impacto económico,"
        "cabe resaltar que los puntos por encima de la línea diagonal indican un ahorro y los que están por debajo indican un sobrecosto.</p></div>")

        f.write("</div>")

        # ── FILA 4 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>Relación entre costos inicial y final en todas las categorías del proceso de bordado</h3>")
        write_html(fig_bord, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>Relación entre el valor costeado y pagado en bordado. El tamaño del punto indica el impacto económico,"
        "cabe resaltar que los puntos por encima de la línea diagonal indican un ahorro y los que están por debajo indican un sobrecosto.</p></div>")

        f.write("<div class='card'><h3>Relación entre costos inicial y final en todas las categorías del proceso de sublimación</h3>")
        write_html(fig_subl, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>Relación entre el valor costeado y pagado en sublimación. El tamaño del punto indica el impacto económico,"
        "cabe resaltar que los puntos por encima de la línea diagonal indican un ahorro y los que están por debajo indican un sobrecosto.</p></div>")

        f.write("</div>")

        f.write("</body></html>")

    size_mb = os.path.getsize("reporte.html") / 1024 / 1024
    print(f"✅ Reporte exportado — Tamaño: {size_mb:.1f} MB")

export_reporte()

✅ Reporte exportado — Tamaño: 1.3 MB
